<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Books for readers

**[Books for readers](https://learning.nextia-ai.com/courses/recommendations/m10/books-for-readers/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 10, case study 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare popularity, item-item neighbours, ALS, a content method on book texts and a hybrid on six million real book ratings, with a split by position in the file and reader segments, and decide what a bookshop should show to whom.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page. The notebook runs in a few minutes on a laptop CPU; making the text vectors the first time adds a few more |
| **Needs** | An internet connection for the setup cells (29 MB of data and a 200 MB text model). No account. Not the course project. |
| **You should know** | Recall@10 and segments, from [Evaluate by segment](https://learning.nextia-ai.com/courses/recommendations/m03/evaluate-by-segment/). Content and hybrids, from [Hybrids](https://learning.nextia-ai.com/courses/recommendations/m06/hybrids/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, matplotlib 3.11.2, implicit 0.7.3, sentence-transformers 6.2.0 and PyTorch 2.15.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab and Kaggle were not run. |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/books-for-readers/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L03-books-for-readers/books-for-readers-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, implicit 0.7.3 (ALS), sentence-transformers 6.2.0 (the text model), and matplotlib. It needs an internet connection and takes one or two minutes the first time.

sentence-transformers needs PyTorch. Colab and Kaggle already have it, so the cell does not install it again. On your own computer, pip installs it (about 100 MB on a Mac or on Windows). On Linux, install the small CPU version first: `pip install torch --index-url https://download.pytorch.org/whl/cpu`.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2 implicit==0.7.3 sentence-transformers==6.2.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy', 'implicit', 'sentence-transformers', 'torch']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It makes a folder `books-for-readers` with a `data` folder in it. It downloads four zip files (29 MB together) **from the dataset's own page on GitHub** and checks each file's checksum (SHA-256). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready:` and the four files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `books-for-readers` (in Colab: the **Files** panel on the left), then run the cell again.

The data is **goodbooks-10k** by Zygmunt Zając ([github.com/zygmuntz/goodbooks-10k](https://github.com/zygmuntz/goodbooks-10k)), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). The course does not copy it: this cell downloads it from its source. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L03-books-for-readers/dataset.md) lists the details. It is real data from Goodreads. Reader IDs are numbers only; do not try to identify anyone.

In [ ]:
# Setup: download the data into books-for-readers/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

SOURCE = "https://github.com/zygmuntz/goodbooks-10k/releases/download/v1.0/"
FILES = {
    "ratings.zip": "2dd0ea8e800860279d7305e88a481ff12090fe5bb54b1321a3854a3b651ebd2e",
    "books.zip": "1e5e29f4037f2c4ccd99cb5f10d07316ec0e3c0fcefbcdf9698585c418f7bd4f",
    "book_tags.zip": "13e9e76e2348676ed61301d1b2cd573c3554cb494cc572c21e99096e1703222f",
    "tags.zip": "02302ad53982627b00d1fff79e48e58dd920f465eb89fb4895eb5ec358bda14f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("books-for-readers").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(SOURCE + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder books-for-readers and run again.")

os.chdir(WORK)
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the case study's tools. Read the comments if you like; the lesson page explains each one.

- `read(folder)` loads the ratings, the books and their tags. `sample(ratings)` keeps one reader in four, so that every step runs in seconds.
- `windows(n)` gives the three windows of the split, as positions in the file. `Period(ratings, start, end)` is the split: everything rated before `start`, and what each reader rated from `start` to `end`. It also gives each reader a segment (new, light or heavy).
- `recommend(...)` and `top_k(...)` turn scores into lists of 10 books. A book the reader already rated is never shown. Readers with no history get a fallback list.
- `evaluate(period, lists, authors, series)` measures the lists: recall@10 with a 95% interval, NDCG@10, coverage, novelty, diversity (different authors), the share from the 100 most read books, the share that continues a series, and recall by segment. The same measures as the course project's evaluation harness, plus two for books.
- `expect(...)` and `expect_hash(...)` check your answers without showing them.

Run the cell. It prints nothing.

In [ ]:
# Helpers: data, the split, lists and measures (the same code as the lesson page).
import re, time, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
warnings.filterwarnings("ignore")

SEGMENTS = ("new", "light", "heavy")


K = 10


N_BOOKS = 10_000


def read(folder):
    """ratings (with their position in the file), books, and each book's tag names in order of use."""
    folder = Path(folder)

    def csv(name):
        with zipfile.ZipFile(folder / f"{name}.zip") as z:
            with z.open(f"{name}.csv") as f:
                return pd.read_csv(f)

    ratings = csv("ratings")
    ratings["pos"] = np.arange(len(ratings))
    books = csv("books").sort_values("book_id").reset_index(drop=True)
    tags = csv("tags").set_index("tag_id")["tag_name"]
    bt = csv("book_tags")
    bt["tag"] = bt["tag_id"].map(tags)
    return ratings, books, bt


def sample(ratings):
    """One reader in four (reader numbers divisible by 4), so that every step runs in seconds on a laptop. Each
    rating keeps its position in the whole file, so the windows stay the same."""
    return ratings[ratings["user_id"] % 4 == 0].reset_index(drop=True)


def windows(ratings_in_file):
    """Positions in ratings.csv: validation 85-90%, test 90-95%, new input 95-100% of the file."""
    n = ratings_in_file
    return {"validation": (int(0.85 * n), int(0.90 * n)), "test": (int(0.90 * n), int(0.95 * n)),
            "new input": (int(0.95 * n), n)}


NOT_TOPICS = re.compile(r"read|own|book|favou?rite|wish|default|kindle|library|buy|borrow|shelf|tbr|series|novels?$"
                        r"|^my-|^i-|audio|e-?book|have|maybe|abandon|dnf|re-|later|19\d\d|20\d\d|^\d")


def topic_tags(bt, books, n=8):
    """The n most used tags of each book that describe the book (not 'to-read', 'owned', 'favorites')."""
    t = bt[~bt["tag"].astype(str).str.contains(NOT_TOPICS)]
    t = t.sort_values(["goodreads_book_id", "count"], ascending=[True, False]).groupby("goodreads_book_id").head(n)
    tags = t.groupby("goodreads_book_id")["tag"].agg(list)
    return books["goodreads_book_id"].map(tags).apply(lambda x: x if isinstance(x, list) else [])


def book_text(row, tags):
    """The text that the embedding model reads: title, author and topic tags."""
    author = str(row.authors).split(",")[0]
    return f"{row.title} by {author}. Tags: {', '.join(tags)}."


def series_of(title):
    """'Catching Fire (The Hunger Games, #2)' -> 'The Hunger Games'; no series -> None."""
    m = re.search(r"\(([^()]*?),?\s*#[\d.\-]+\)\s*$", str(title))
    return m.group(1).strip() if m else None


class Period:
    """Everything rated before row `start` of ratings.csv, and what readers rated in rows start to end - 1."""

    def __init__(self, ratings, start, end, recent=300_000):
        self.start, self.end = int(start), int(end)
        before = ratings[ratings["pos"] < self.start]
        self.users = np.array(sorted(before["user_id"].unique()))
        self.uidx = {u: i for i, u in enumerate(self.users)}
        rows = before["user_id"].map(self.uidx).to_numpy()
        cols = before["book_id"].to_numpy() - 1
        shape = (len(self.users), N_BOOKS)
        self.X = sp.csr_matrix((np.ones(len(before), np.float32), (rows, cols)), shape=shape)  # 1 = rated
        self.R = sp.csr_matrix((before["rating"].to_numpy().astype(np.float32), (rows, cols)), shape=shape)
        # Order of each rating in the reader's history (1 = first): newer books weigh more in profiles.
        self.O = sp.csr_matrix(((before.groupby("user_id").cumcount() + 1).to_numpy().astype(np.float32),
                                (rows, cols)), shape=shape)
        self.item_readers = np.asarray(self.X.sum(0)).ravel()
        self.available = self.item_readers > 0
        last = before[before["pos"] >= self.start - recent]
        self.recent_readers = np.bincount(last["book_id"].to_numpy() - 1, minlength=N_BOOKS).astype(float)
        inside = ratings[(ratings["pos"] >= self.start) & (ratings["pos"] < self.end)]
        self.truth = {u: set(g - 1) for u, g in inside.groupby("user_id")["book_id"]}
        self.readers = np.array(sorted(self.truth))
        n = np.array([self.X[self.uidx[u]].nnz if u in self.uidx else 0 for u in self.readers])
        self.history = n
        self.segment = np.where(n == 0, "new", np.where(n < 50, "light", "heavy"))
        self.liked_in_window = inside.groupby("user_id")["rating"].apply(lambda r: (r >= 4).mean())


def top_k(scores, k=K):
    """The k best columns of each row, best first."""
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, part, axis=1), axis=1, kind="stable")
    return np.take_along_axis(part, order, axis=1)


def recommend(period, scores_of, fallback, k=K, batch=2000):
    """Lists of k books for every reader of the period. Books the reader already rated are never shown (a reader
    rates a book once). Readers with no history get the fallback list."""
    rows = np.array([period.uidx.get(u, -1) for u in period.readers])
    lists = np.tile(np.asarray(fallback[:k]), (len(rows), 1))
    known = np.flatnonzero(rows >= 0)
    for i in range(0, len(known), batch):
        idx = known[i:i + batch]
        r = rows[idx]
        s = np.asarray(scores_of(r), dtype=np.float64)
        s[~np.isfinite(s)] = -np.inf
        seen = period.X[r]
        s[seen.nonzero()] = -np.inf          # already rated
        s[:, ~period.available] = -np.inf     # never rated before the cut: nothing to learn from
        lists[idx] = top_k(s, k)
    return lists


def timed(f, *args, **kwargs):
    t0 = time.perf_counter()
    out = f(*args, **kwargs)
    return out, time.perf_counter() - t0


def bootstrap(values, n=1000, seed=37):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n, len(values)))].mean(axis=1)
    return float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def per_reader(period, lists, k=K):
    """recall@k and NDCG@k for each reader."""
    disc = 1 / np.log2(np.arange(2, k + 2))
    rec, ndcg = np.zeros(len(lists)), np.zeros(len(lists))
    for i, u in enumerate(period.readers):
        truth = period.truth[u]
        hits = np.isin(lists[i, :k], list(truth)).astype(float)
        rec[i] = hits.sum() / len(truth)
        ndcg[i] = (hits * disc).sum() / disc[:min(len(truth), k)].sum()
    return rec, ndcg


def evaluate(period, lists, authors, series, k=K):
    """The harness's measures for one method: accuracy, beyond accuracy, and recall by segment."""
    rec, ndcg = per_reader(period, lists, k)
    L = lists[:, :k]
    info = -np.log2((period.item_readers + 1) / (len(period.users) + 1))
    top100 = np.argsort(-period.item_readers, kind="stable")[:100]
    a = authors[L]
    diversity = 1 - ((a[:, :, None] == a[:, None, :]).sum((1, 2)) - k) / (k * (k - 1))
    # A suggestion "continues a series" when the reader already rated another book of the same series.
    started = []
    for i, u in enumerate(period.readers):
        r = period.uidx.get(u, -1)
        own = {series[b] for b in period.X[r].indices} - {None} if r >= 0 else set()
        started.append(np.mean([series[b] in own for b in L[i]]) if own else 0.0)
    out = {
        "recall": bootstrap(rec), "ndcg": float(ndcg.mean()),
        "coverage": float(np.unique(L).size / period.available.sum()),
        "novelty": float(info[L].mean()), "diversity": float(diversity.mean()),
        "top100_share": float(np.isin(L, top100).mean()), "series_share": float(np.mean(started)),
        "segments": {s: bootstrap(rec[period.segment == s]) for s in SEGMENTS},
    }
    return out, rec


def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A small online bookshop wants a block of ten books on each reader's home page: "Books for you". Mina, who chooses the books for the shop, says: "Please do not show Harry Potter to everybody. Our readers know it already." Leo, the shop's developer, asks: "Which method, for whom, and what does it cost to run?" Mina, Leo and the bookshop are made up; the ratings are real, from Goodreads.

Your job: compare popularity, item-item neighbours, ALS, a content method on book texts, and a hybrid, on a split by position in the file, by reader segment. Look at accuracy **and** at coverage and novelty, against the popularity baseline. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/books-for-readers/#the-situation) tells the full story.

## 2. The data

Run the next cell. It reads the ratings, the books and their tags, and keeps one reader in four (reader numbers divisible by 4), so that every step is fast.

> **Predict.** The 100 most read books are 1% of the 10,000 books. What share of all ratings do they get: about 1%, about 5%, or more than 15%?

> **Check.** You should see 5,976,479 ratings from 53,424 readers, and 1,493,771 ratings from 13,356 readers in the sample.

In [ ]:
full, books, book_tags = read("data")
print(f"{len(full):,} ratings, {full.user_id.nunique():,} readers, {full.book_id.nunique():,} books")
top100 = full.book_id.value_counts().head(100)
print(f"the 100 most read books get {top100.sum() / len(full):.1%} of the ratings")
ratings = sample(full)
print(f"sample: {len(ratings):,} ratings from {ratings.user_id.nunique():,} readers")
authors = np.array([str(a).split(",")[0] for a in books.authors])
series = np.array([series_of(t) for t in books.title], dtype=object)
books[["book_id", "title", "authors", "original_publication_year"]].head()

**What this tells us.** A few books get a large part of all ratings. A method that follows the counts will show the same famous books to everybody: the "Harry Potter problem" that Mina is afraid of.

## 3. The split by position, and the segments

The data has **no dates**. The README says that the ratings are sorted by time, and the whole file is in one order. So we use the position in the file as the clock: the methods learn from the ratings before a position, and we score them on the ratings in the next window.

- validation: 85% to 90% of the file (to choose settings),
- test: 90% to 95% (each method once),
- new input: 95% to 100% (the chosen design, once).

Each reader in a window gets a segment from their ratings before it: **new** (0), **light** (1 to 49) or **heavy** (50 or more).

> **Check.** You should see 5,591 readers in the test window: 180 new, 535 light and 4,876 heavy.

In [ ]:
W = windows(len(full))
test = Period(ratings, *W["test"])
print(f"{len(test.readers):,} readers rated books in the test window")
print(pd.Series(test.segment).value_counts().reindex(SEGMENTS).to_string())
print(f"learn from: {len(test.users):,} readers x {int(test.available.sum()):,} books rated before the window")

> **Your turn.** What share of the (reader, book) pairs in the test window are books first published in 2015 or later? Use `test.truth` (reader → set of book positions, 0 to 9,999) and `books.original_publication_year` (row `i` is book position `i`). Store the share in `share_new_books`.

In [ ]:
# Your answer: the share of test-window pairs that are books from 2015 or later.
share_new_books = ...

In [ ]:
expect('share_new_books', rounded(share_new_books), 0.164, tolerance=0.002)

**What this tells us.** Near the end of the file, readers rate many books that were new at the time. A method that learns from old ratings knows little about them. Keep this in mind for the baselines.

## 4. The baselines: popular and trending

Run the next cell. `popular` ranks books by the number of readers before the window. `trending` does the same for the last 300,000 ratings of the file before the window (about 5% of the file). Everyone gets the same list, minus the books they already rated.

In [ ]:
def popular(period):
    """The books with the most readers before the cut."""
    return np.argsort(-period.item_readers, kind="stable")


def trending(period):
    """The books with the most readers in the last 300,000 ratings before the cut (about 5% of the file)."""
    return np.argsort(-period.recent_readers, kind="stable")


def popular_scores(period, counts):
    """The same popularity list as scores for every reader (so that `recommend` removes books they read)."""
    return lambda rows: np.tile(np.log1p(counts), (len(rows), 1))


results, lists = {}, {}
lists["popular"] = recommend(test, popular_scores(test, test.item_readers), popular(test))
lists["trending"] = recommend(test, popular_scores(test, test.recent_readers), trending(test))
for name in lists:
    results[name], _ = evaluate(test, lists[name], authors, series)
    print(f"{name:10} recall@10 {results[name]['recall'][0]:.3f}")
print("trending now:", [books.title[b][:30] for b in trending(test)[:5]])

> **Check.** You should see popular 0.015 and trending 0.044.

## 5. Neighbours, and the Harry Potter problem

Run the next cell. **Co-occurrence** counts the readers who rated both books. **Item-cf** divides that count by the size of each book (cosine), as in [People also bought](https://learning.nextia-ai.com/courses/recommendations/m04/people-also-bought/). Both keep the 100 nearest books of each book, and score a book for a reader by adding its similarity to each book they rated. It prints the 5 nearest books of *The Martian* both ways. It takes about 20 seconds.

> **Predict.** By count, which books will be nearest to *The Martian*, a science-fiction novel about an astronaut?

In [ ]:
def neighbours(period, measure="cosine", n=100, block=1000):
    """The n nearest books of every book. 'count' = readers who rated both; 'cosine' = count / sqrt(nA x nB).
    Computed in blocks of books so that the full 10,000 x 10,000 table is never in memory."""
    XT = period.X.T.tocsr()
    norm = np.sqrt(period.item_readers) + 1e-9
    rows, cols, vals = [], [], []
    for a in range(0, N_BOOKS, block):
        b = min(a + block, N_BOOKS)
        C = (XT[a:b] @ period.X).toarray()                 # shared readers: books a..b x all books
        C[np.arange(b - a), np.arange(a, b)] = 0           # a book is not its own neighbour
        if measure == "cosine":
            C = C / norm[a:b, None] / norm[None, :]
        top = top_k(C, n)
        rows.append(np.repeat(np.arange(a, b), n))
        cols.append(top.ravel())
        vals.append(np.take_along_axis(C, top, axis=1).ravel())
    S = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(N_BOOKS, N_BOOKS))
    S.eliminate_zeros()
    return S


def neighbour_scores(period, S, rows):
    """Sum of the similarities of each book to the books the reader rated."""
    return (period.X[rows] @ S).toarray()


C = neighbours(test, "count")
S = neighbours(test, "cosine")
martian = int(np.flatnonzero(books.title.to_numpy() == "The Martian")[0])
for name, M in (("count", C), ("cosine", S)):
    near = np.argsort(-M[martian].toarray().ravel(), kind="stable")[:5]
    print(f"{name:7}", " | ".join(books.title[b][:32] for b in near))
lists["co-occurrence"] = recommend(test, lambda rows: neighbour_scores(test, C, rows), trending(test))
lists["item-cf"] = recommend(test, lambda rows: neighbour_scores(test, S, rows), trending(test))
for name in ("co-occurrence", "item-cf"):
    results[name], _ = evaluate(test, lists[name], authors, series)
    print(f"{name:14} recall@10 {results[name]['recall'][0]:.3f}  top-100 share {results[name]['top100_share']:.0%}")

> **Check.** By count, the first neighbour is *The Hunger Games (The Hunger Games, #1)*; by cosine, *Ready Player One*. Recall@10: co-occurrence 0.016, item-cf 0.024.

> **Your turn.** In what share of the co-occurrence lists is there at least one Harry Potter book? `series[b]` is the series of book `b` (`"Harry Potter"` for the Harry Potter books). Store the share in `harry_potter_share`.

In [ ]:
# Your answer: the share of co-occurrence lists with at least one Harry Potter book.
harry_potter_share = ...

In [ ]:
expect('harry_potter_share', rounded(harry_potter_share), 0.834, tolerance=0.002)

**What this tells us.** Counting puts the famous books next to everything. Cosine fixes much of it, as in the course's lesson. The shop still needs to know: does that cost accuracy?

## 6. ALS

Run the next cell. ALS learns a short vector for each reader and each book, as in [Hidden tastes](https://learning.nextia-ai.com/courses/recommendations/m05/hidden-tastes/). We use the settings that were best on the **validation** window: 128 factors and regularisation 10. The grid of 9 settings is on the lesson page. The fit takes about 27 seconds on one thread.

In [ ]:
def fit_als(period, factors=64, regularization=1.0, alpha=1.0, iterations=15, seed=37):
    import implicit
    from threadpoolctl import threadpool_limits
    conf = (period.X * alpha).astype(np.float32).tocsr()
    with threadpool_limits(1):
        model = implicit.cpu.als.AlternatingLeastSquares(factors=factors, regularization=regularization,
                                                         iterations=iterations, random_state=seed, num_threads=1)
        model.fit(conf, show_progress=False)
    return model


def als_scores(model, rows):
    return model.user_factors[rows] @ model.item_factors.T


ALS = {"factors": 128, "regularization": 10.0}
als = fit_als(test, **ALS)
lists["als"] = recommend(test, lambda rows: als_scores(als, rows), trending(test))
results["als"], _ = evaluate(test, lists["als"], authors, series)
print(f"als  recall@10 {results['als']['recall'][0]:.3f}")

> **Check.** You should see 0.034.

## 7. Content: books with similar texts

A content method needs no other readers. It compares the **texts** of the books, as in [Content-based recommendations](https://learning.nextia-ai.com/courses/recommendations/m06/content-based-recommendations/). Each book's text is its title, its first author and up to 8 tags that describe it (we drop shelves such as `to-read` and `owned`).

Run the next cell. It downloads the course's text model, IBM Granite Embedding 97M multilingual R2 (Apache-2.0, about 200 MB, once), and turns the 10,000 texts into vectors of 384 numbers. This takes a few minutes on a laptop CPU, and longer on a slow or busy computer. The cell saves the vectors in `data/book_vectors.npy`; when you run it again, it loads them instead.

> **Check.** You should see the text of book 2 and the shape `(10000, 384)`.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
tags = topic_tags(book_tags, books)
texts = [book_text(row, t) for row, t in zip(books.itertuples(), tags)]
print(texts[1])
saved = Path("data/book_vectors.npy")   # kept by an earlier run of this cell
if saved.exists():
    emb = np.load(saved)
    print("loaded the vectors saved by an earlier run")
else:
    model = SentenceTransformer("ibm-granite/granite-embedding-97m-multilingual-r2", revision="835ad14087e140460703cf0fae09f97d469d65c2", device="cpu")
    start = time.perf_counter()
    emb = model.encode(texts, batch_size=64, normalize_embeddings=True).astype(np.float32)
    np.save(saved, emb)
    print(f"encoded in {time.perf_counter() - start:.0f} s")
print(emb.shape)

Run the next cell. A reader's **profile** is the mean vector of their last 20 books rated 4 or 5. The score of a book is its cosine with the profile. It also prints the 5 books whose texts are nearest to *The Martian*.

In [ ]:
def profiles(period, emb, rows, last=20):
    """A reader's taste as one vector: the mean embedding of their last `last` books rated 4 or 5 (or of their last
    books, if they rated none 4 or 5), scaled to length 1."""
    P = np.zeros((len(rows), emb.shape[1]), np.float32)
    for i, r in enumerate(rows):
        books, order = period.O[r].indices, period.O[r].data       # the reader's books and their order (1 = first)
        liked = period.R[r].toarray().ravel()[books] >= 4
        if liked.any():
            books, order = books[liked], order[liked]
        P[i] = emb[books[np.argsort(-order)[:last]]].mean(axis=0)  # the newest `last` books
    return P / np.maximum(np.linalg.norm(P, axis=1, keepdims=True), 1e-9)


def content_scores(period, emb, rows):
    """Cosine between each book's embedding and the reader's profile."""
    return profiles(period, emb, rows) @ emb.T


s = emb @ emb[martian]
s[martian] = -np.inf
print("text   ", " | ".join(books.title[b][:32] for b in np.argsort(-s)[:5]))
lists["content"] = recommend(test, lambda rows: content_scores(test, emb, rows), trending(test))
results["content"], _ = evaluate(test, lists["content"], authors, series)
print(f"content  recall@10 {results['content']['recall'][0]:.3f}  series share {results['content']['series_share']:.0%}")

> **Check.** You should see content recall@10 close to 0.018. (The text model can give slightly different vectors on another computer, so the third decimal can differ.)

## 8. A hybrid

Run the next cell. The hybrid adds the ALS, content and trending scores, each scaled from 0 to 1 for each reader, as in [Hybrids](https://learning.nextia-ai.com/courses/recommendations/m06/hybrids/). We chose the weights on the validation window: ALS 1, content 1, trending 2.

In [ ]:
def scale(s):
    """Each row scaled to 0..1 (as the course project's hybrid)."""
    lo, hi = s.min(axis=1, keepdims=True), s.max(axis=1, keepdims=True)
    return (s - lo) / np.where(hi > lo, hi - lo, 1)


def hybrid_scores(period, model, emb, rows, w_als=1.0, w_content=1.0, w_trending=0.0):
    """A weighted sum of ALS, content and trending scores, each scaled 0..1 for each reader."""
    t = np.tile(np.log1p(period.recent_readers), (len(rows), 1))
    return (w_als * scale(als_scores(model, rows)) + w_content * scale(content_scores(period, emb, rows))
            + w_trending * scale(t))


WEIGHTS = {"w_als": 1.0, "w_content": 1.0, "w_trending": 2.0}
lists["hybrid"] = recommend(test, lambda rows: hybrid_scores(test, als, emb, rows, **WEIGHTS), trending(test))
results["hybrid"], _ = evaluate(test, lists["hybrid"], authors, series)
print(f"hybrid  recall@10 {results['hybrid']['recall'][0]:.3f}")

> **Check.** You should see about 0.057.

## 9. Compare, by segment

Run the next cell. It shows every method on the same readers, with the same measures. "top 100" is the share of suggestions from the 100 most read books; "series" is the share that continues a series the reader started.

In [ ]:
def table(results):
    """One row per method: recall@10 (95% interval), NDCG@10, beyond-accuracy measures, recall by segment."""
    rows = []
    for name, o in results.items():
        rows.append({"method": name, "recall@10": round(o["recall"][0], 3),
                     "interval": f"{o['recall'][1]:.3f} to {o['recall'][2]:.3f}", "NDCG@10": round(o["ndcg"], 3),
                     "coverage": f"{o['coverage']:.1%}", "novelty": round(o["novelty"], 2),
                     "top 100": f"{o['top100_share']:.0%}", "series": f"{o['series_share']:.0%}",
                     **{s: round(o["segments"][s][0], 3) for s in SEGMENTS}})
    return pd.DataFrame(rows).set_index("method")


table(results)

Compare the methods on the **same readers**, as in [Evaluate by segment](https://learning.nextia-ai.com/courses/recommendations/m03/evaluate-by-segment/).

> **Your turn.** `evaluate` also returns recall@10 for each reader (its second value). (Your text vectors can differ a little from the course's, so the check accepts a count within 5%.) Compute it for `lists["hybrid"]` and `lists["trending"]`. Store in `better` the number of readers for whom the hybrid has the higher recall@10.

In [ ]:
# Your answer: for how many readers is the hybrid better than trending?
better = ...

In [ ]:
expect('better', better, 822.0, tolerance=0.05)

**What this tells us.** Compare the result with the lesson page: the mean difference is +0.014, with a 95% interval from +0.011 to +0.017.

## 10. End to end on new input: the last 5% of the file

The last window was never used. Run the next cell. It does the whole pipeline again, one window later: ALS learns on everything before 95% of the file, and the hybrid ranks for the last window with the same weights. New readers get trending.

> **Check.** You should see 5,568 readers and hybrid recall@10 close to 0.055 (trending 0.041).

In [ ]:
last = Period(ratings, *W["new input"])
als_last = fit_als(last, **ALS)
final = {"trending": recommend(last, popular_scores(last, last.recent_readers), trending(last)),
         "hybrid": recommend(last, lambda rows: hybrid_scores(last, als_last, emb, rows, **WEIGHTS), trending(last))}
print(f"{len(last.readers):,} readers rated books in the last window")
table({name: evaluate(last, l, authors, series)[0] for name, l in final.items()})

Run the next cell. It prints one light reader's last books, then their hybrid list, with a mark on each book they rated in the last window.

In [ ]:
reader = 1260
i = list(last.readers).index(reader)
row = last.uidx[reader]
order = last.O[row].toarray().ravel()
print("last rated:", " | ".join(books.title[b][:30] for b in np.argsort(-order)[:5]))
for place, b in enumerate(final["hybrid"][i], 1):
    mark = "rated" if b in last.truth[reader] else "     "
    print(f"{place:2}. {mark}  {books.title[b][:50]} ({authors[b]})")

> **Check.** You should see 4 book(s) marked `rated`.

## 11. Recap

You split real ratings by their position in the file, and compared popularity, co-occurrence, item-cf, ALS, a content method and a hybrid on the same readers, by segment, with coverage, novelty and the share of famous books. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/books-for-readers/#the-recommendation) turns this into a recommendation with its costs, its limits and what a person still checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [What to learn next](https://learning.nextia-ai.com/courses/recommendations/m10/what-to-learn-next/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m10/books-for-readers/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real: goodbooks-10k by Zygmunt Zając, CC BY-SA 4.0, downloaded from its source. Reader IDs are numbers only.